# Customer Churn — Data Quality and Exploratory Analysis

**Objective:** Audit and clean the supplied synthetic customer churn dataset, document quality decisions, and inspect initial churn patterns.

**Decision question:** Which customer groups should a fictional subscription team contact first to reduce avoidable churn?

> Findings are descriptive. They do not establish causation.


## 1. Load the supplied dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("customer_churn_sample.csv")
print("Rows:", len(df))
print("Columns:", df.shape[1])
display(df.head())
display(df.dtypes.to_frame("Data Type"))


## 2. Profile missing values and duplicates

In [ ]:
profile = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique()
})
display(profile)
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate CustomerID:", df["CustomerID"].duplicated().sum())


## 3. Check numeric extremes using the 1.5×IQR rule

In [ ]:
numeric_cols = ["Age","TenureMonths","MonthlyCharges","TotalCharges","SupportTickets"]

rows = []
for col in numeric_cols:
    q1 = df[col].quantile(.25)
    q3 = df[col].quantile(.75)
    iqr = q3 - q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    rows.append([col,q1,q3,iqr,lower,upper,count])

outliers = pd.DataFrame(rows, columns=["Variable","Q1","Q3","IQR","Lower Bound","Upper Bound","Outlier Count"])
display(outliers)


## 4. Logical and category validation

In [ ]:
checks = {
    "CustomerID format valid": df["CustomerID"].astype(str).str.match(r"^CUST-\d+$").all(),
    "Age > 0": (df["Age"] > 0).all(),
    "TenureMonths >= 0": (df["TenureMonths"] >= 0).all(),
    "MonthlyCharges > 0": (df["MonthlyCharges"] > 0).all(),
    "TotalCharges >= 0": (df["TotalCharges"] >= 0).all(),
    "SupportTickets >= 0": (df["SupportTickets"] >= 0).all(),
    "Churn values valid": df["Churn"].isin(["Yes","No"]).all(),
    "TotalCharges consistent with MonthlyCharges × TenureMonths":
        np.allclose(df["TotalCharges"], df["MonthlyCharges"]*df["TenureMonths"], atol=.02)
}
for col, values in {
    "Gender":["Female","Male"],
    "SubscriptionType":["Basic","Pro","Enterprise"],
    "ContractType":["Month-to-Month","One Year","Two Year"],
    "PaymentMethod":["Credit Card","Bank Transfer","UPI","Debit Card"],
    "Churn":["Yes","No"]
}.items():
    checks[f"{col} categories valid"] = df[col].isin(values).all()

display(pd.DataFrame({"Check":checks.keys(),"Passed":checks.values()}))


## 5. Cleaning log

| Issue | Finding | Action | Reason |
|---|---|---|---|
| Missing values | None | No change | No fields are missing |
| Duplicate rows | 0 | No deletion | No exact duplicate records |
| Duplicate CustomerID | 0 | No deletion | Every ID is unique |
| Invalid categories | None | No change | Values match expected categories |
| Impossible numeric values | None | No change | Logical validation passed |
| IQR outliers | 0 | No removal | No numeric values exceed 1.5×IQR bounds |
| Charge consistency | Passed | No change | TotalCharges matches MonthlyCharges × TenureMonths within ₹0.02 |

No rows were silently deleted or imputed.


In [ ]:
# Produce the analysis-ready dataset
cleaned = df.copy()
cleaned.to_csv("customer_churn_cleaned.csv", index=False)

# Post-cleaning validation
assert cleaned.shape == df.shape
assert cleaned.isna().sum().sum() == 0
assert cleaned["CustomerID"].duplicated().sum() == 0
assert np.allclose(
    cleaned["TotalCharges"],
    cleaned["MonthlyCharges"] * cleaned["TenureMonths"],
    atol=.02
)
print("Post-cleaning validation: PASSED")


## 6. Summary statistics

In [ ]:
display(df[numeric_cols].describe().T)
print(f"Overall churn rate: {(df['Churn'].eq('Yes').mean()*100):.1f}%")


## 7. Meaningful distributions

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(df["Age"], bins=6)
ax.set_title("Age Distribution")
ax.set_xlabel("Age")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(df["TenureMonths"], bins=6)
ax.set_title("Tenure Distribution")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(df["SupportTickets"], bins=7)
ax.set_title("Support Ticket Distribution")
ax.set_xlabel("Tickets in previous 90 days")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()


## 8. Churn by meaningful customer groups

In [ ]:
def churn_summary(column):
    result = df.groupby(column).agg(
        Customers=("CustomerID","count"),
        Churned=("Churn", lambda s: (s=="Yes").sum())
    )
    result["Churn Rate"] = result["Churned"]/result["Customers"]
    return result.sort_values("Churn Rate", ascending=False)

display(churn_summary("ContractType"))
display(churn_summary("SubscriptionType"))
display(churn_summary("SupportTickets"))


## 9. Tenure bands

In [ ]:
analysis = df.copy()
analysis["TenureBand"] = pd.cut(
    analysis["TenureMonths"],
    bins=[-1,6,12,24,np.inf],
    labels=["0–6 months","7–12 months","13–24 months","25+ months"]
)
tenure = analysis.groupby("TenureBand", observed=True).agg(
    Customers=("CustomerID","count"),
    Churned=("Churn", lambda s: (s=="Yes").sum())
)
tenure["Churn Rate"] = tenure["Churned"]/tenure["Customers"]
display(tenure)


## 10. Decision-oriented recommendation

**Audience:** Month-to-month customers with **3 or more support tickets** in the previous 90 days.

**Decision:** Contact this group first with a retention/support follow-up.

**Supporting metric:** The segment contains **7 customers**, and **7 churned**, giving an observed churn rate of **100.0%**, compared with **46.7% overall churn**.

**Uncertainty:** The sample has only 15 customers and is synthetic, so the segment percentage is unstable. The result is an association, not evidence that month-to-month contracts or support tickets cause churn.

**Next measurement:** After outreach, measure retention/churn for the contacted segment and compare it with an appropriate comparison group on a larger sample.


## 11. Reproducibility notes

- Source: `customer_churn_sample.csv`
- Output: `customer_churn_cleaned.csv`
- No rows were removed.
- Missing values, duplicate IDs, invalid categories, impossible values, IQR outliers, and charge consistency were explicitly checked.
